# Week 6 Exercise — The Price Is Right: Product Price Predictor

**By:** Jaymineh  
**API:** OpenAI (fine-tuning GPT-4o-mini) + OpenRouter (zero-shot pricing)

A product price predictor that compares traditional ML, zero-shot LLM pricing, and
fine-tuned model approaches using Amazon product data.

**Week 6 skills demonstrated:**
- Loading curated product data from HuggingFace Hub
- Traditional ML baseline with **XGBoost** (feature engineering + NLP)
- **Zero-shot LLM pricing** via OpenRouter
- **Fine-tuning GPT-4o-mini** via OpenAI API with JSONL training data
- Evaluation using MAE, scatter plots, and the `pricer.evaluator` framework
- Comparing all approaches side by side

## Step 1: Setup & Load Data

In [ ]:
import sys
from pathlib import Path

# Add week6 root to path so pricer module is importable
week6_dir = str(Path.cwd().parent.parent)
if week6_dir not in sys.path:
    sys.path.insert(0, week6_dir)
print(f"Added to path: {week6_dir}")

In [ ]:
import os
import json
import re
import time
import random
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from litellm import completion
import plotly.graph_objects as go

load_dotenv(override=True)

openai_key = os.getenv('OPENAI_API_KEY')
openrouter_key = os.getenv('OPENROUTER_API_KEY')
print(f"OpenAI key: {'YES' if openai_key else 'NO'}")
print(f"OpenRouter key: {'YES' if openrouter_key else 'NO'}")

In [ ]:
from pricer.items import Item
from pricer.evaluator import Tester, evaluate

train, val, test = Item.from_hub('ed-donner/items_lite')
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")
print(f"\nSample item: {train[0].title}")
print(f"Price: ${train[0].price}")
print(f"Summary: {train[0].summary[:200]}...")

In [ ]:
# Quick EDA: price distribution
prices = [item.price for item in train]
print(f"Price range: ${min(prices):.2f} - ${max(prices):.2f}")
print(f"Mean: ${np.mean(prices):.2f}")
print(f"Median: ${np.median(prices):.2f}")
print(f"Std: ${np.std(prices):.2f}")

## Step 2: Baselines & Traditional ML (Day 3 Concepts)

In [ ]:
# Baseline 1: Always predict the mean price
average_price = np.mean([item.price for item in train])

def constant_pricer(item):
    return average_price

print(f"Constant prediction: ${average_price:.2f}")
evaluate(constant_pricer, test, size=200)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import CountVectorizer

# Build text features from summaries
train_texts = [item.summary or item.title for item in train]
train_prices = np.array([item.price for item in train])

vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X_train = vectorizer.fit_transform(train_texts)

# Train Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, train_prices)
print(f"Random Forest trained on {X_train.shape[0]} items with {X_train.shape[1]} features")

def random_forest_pricer(item):
    text = item.summary or item.title
    X = vectorizer.transform([text])
    return rf.predict(X)[0]

evaluate(random_forest_pricer, test, size=200)

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1,
)
xgb.fit(X_train, train_prices)
print(f"XGBoost trained")

def xgboost_pricer(item):
    text = item.summary or item.title
    X = vectorizer.transform([text])
    return xgb.predict(X)[0]

evaluate(xgboost_pricer, test, size=200)

## Step 3: Zero-Shot LLM Pricing (Day 4 Concepts)

Ask an LLM to estimate the price from the product description alone, with no training.

In [ ]:
def zero_shot_gpt4o_mini(item):
    text = item.summary or item.title
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{text}"
    try:
        response = completion(
            model="openrouter/openai/gpt-4o-mini",
            messages=[{"role": "user", "content": message}],
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error: {e}")
        return str(average_price)

# Test on one item first
sample = test[0]
result = zero_shot_gpt4o_mini(sample)
print(f"Product: {sample.title[:60]}...")
print(f"Actual: ${sample.price:.2f}")
print(f"Predicted: {result}")

In [ ]:
# Run on 200 test items (takes ~2-3 min with threading)
evaluate(zero_shot_gpt4o_mini, test, size=200)

## Step 4: Fine-Tune GPT-4o-mini (Day 5 Concepts)

Create JSONL training data, upload to OpenAI, fine-tune, and evaluate.

In [ ]:
# Create JSONL training and validation data
TRAIN_SIZE = 400
VAL_SIZE = 100

def make_jsonl_entry(item):
    text = item.summary or item.title
    return {
        "messages": [
            {"role": "user", "content": f"Estimate the price of this product. Respond with the price, no explanation\n\n{text}"},
            {"role": "assistant", "content": f"${item.price:.2f}"}
        ]
    }

# Sample training and validation data
random.seed(42)
train_sample = random.sample(train, TRAIN_SIZE)
val_sample = random.sample(val, VAL_SIZE)

train_jsonl_path = "fine_tune_train.jsonl"
val_jsonl_path = "fine_tune_val.jsonl"

for path, data in [(train_jsonl_path, train_sample), (val_jsonl_path, val_sample)]:
    with open(path, "w") as f:
        for item in data:
            f.write(json.dumps(make_jsonl_entry(item)) + "\n")

print(f"Created {train_jsonl_path} with {TRAIN_SIZE} examples")
print(f"Created {val_jsonl_path} with {VAL_SIZE} examples")

# Preview one entry
sample_entry = make_jsonl_entry(train_sample[0])
print(f"\nSample entry:")
print(json.dumps(sample_entry, indent=2)[:500])

In [ ]:
# Upload files to OpenAI
openai_client = OpenAI()

with open(train_jsonl_path, "rb") as f:
    train_file = openai_client.files.create(file=f, purpose="fine-tune")
print(f"Train file uploaded: {train_file.id}")

with open(val_jsonl_path, "rb") as f:
    val_file = openai_client.files.create(file=f, purpose="fine-tune")
print(f"Val file uploaded: {val_file.id}")

In [ ]:
# Start fine-tuning job
ft_job = openai_client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="gpt-4o-mini-2024-07-18",
    hyperparameters={
        "n_epochs": 1,
    },
)
print(f"Fine-tuning job created: {ft_job.id}")
print(f"Status: {ft_job.status}")

In [ ]:
# Poll for completion (fine-tuning typically takes 5-15 min)
while True:
    job = openai_client.fine_tuning.jobs.retrieve(ft_job.id)
    print(f"Status: {job.status}")
    if job.status in ["succeeded", "failed", "cancelled"]:
        break
    time.sleep(30)

if job.status == "succeeded":
    fine_tuned_model = job.fine_tuned_model
    print(f"\nFine-tuned model: {fine_tuned_model}")
else:
    print(f"\nJob ended with status: {job.status}")
    if job.error:
        print(f"Error: {job.error}")

In [ ]:
# Evaluate the fine-tuned model
def fine_tuned_pricer(item):
    text = item.summary or item.title
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{text}"
    try:
        response = openai_client.chat.completions.create(
            model=fine_tuned_model,
            messages=[{"role": "user", "content": message}],
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error: {e}")
        return str(average_price)

# Test on one item
sample = test[0]
result = fine_tuned_pricer(sample)
print(f"Product: {sample.title[:60]}...")
print(f"Actual: ${sample.price:.2f}")
print(f"Fine-tuned prediction: {result}")

In [ ]:
# Full evaluation on 200 test items
evaluate(fine_tuned_pricer, test, size=200)

## Step 5: Compare All Approaches

In [ ]:
# After running all evaluations, manually record the MAE results here
# Update these values with actual results from the evaluations above

results = {
    "Constant (Mean)": 106.08,       # Update after running
    "Random Forest": 68.07,          # Update after running
    "XGBoost": 77.34,                # Update after running
    "GPT-4o-mini (zero-shot)": 47.89, # Update after running
    "GPT-4o-mini (fine-tuned)": 65.21,  # From Fine Tuned Pricer (MAE)
}

# Filter out None values (in case some haven't been run yet)
results = {k: v for k, v in results.items() if v is not None}

if results:
    fig = go.Figure(data=[go.Bar(
        x=list(results.keys()),
        y=list(results.values()),
        text=[f"${v:.2f}" for v in results.values()],
        textposition='outside',
        marker_color=['#636EFA', '#636EFA', '#636EFA', '#EF553B', '#00CC96'],
    )])
    fig.update_layout(
        title="Mean Absolute Error by Approach (lower is better)",
        yaxis_title="MAE ($)",
        width=900,
        height=500,
    )
    fig.show()
else:
    print("Run the evaluations above first, then fill in the results dict.")

## Summary

| Approach | MAE | Notes |
|----------|-----|-------|
| Constant (Mean) | 106.08 | Always predicts average price |
| Random Forest | 68.07 | 100 trees, CountVectorizer 2000 features |
| XGBoost | 77.34 | 500 trees, learning_rate=0.05 |
| GPT-4o-mini (zero-shot) | 47.89 | Via OpenRouter, no training |
| GPT-4o-mini (fine-tuned) | 65.21 | 400 train, 100 val, 1 epoch |

### Key Takeaways
- Traditional ML models provide a strong baseline without API costs
- Zero-shot LLMs can price products reasonably well using general knowledge
- Fine-tuning on domain data teaches the model price patterns specific to this dataset
- The evaluation framework (MAE + scatter plots) enables systematic comparison